In [13]:
from typing import List, Tuple, Union, Dict, Optional
import sympy as sp
import numpy as np
from numpy import ndarray
from sympy import symbols, cos, sin, tan, Matrix, pprint
import scipy

# Quadrotor 12 State Dynamic Linearization
One popular manner of stabilizing a nonlinear system is to linearize it about some equilibrium point and use LQR to produce a linear state-feedback controller that stabilizes the system to this equilibrium point. For a linear time-invariant (LTI) where the goal is to stabilize the system with an infinite time-horizon, the LQR problem has an exact solution given by the algebraic Ricatti equation (ARE).

Below, we symbolically define the parameters of our system. In the paper, we are only given the equations on how to update our system's positional and angular acceleration. However, if we want our system to be linearized in the form:
$$
\dot{x} = Ax + Bu
$$
then it is quite easy to deduce what the element in our $A$ and $B$ matrices should look like for position and velocity. Therefore, we only need to linearize the positional and angular acceleration using Sympy's Jacobian operator, and then manually fill in our $A$ and $B$ matrices.

In [14]:
# Define state variables and control input
m, g, km, kf = sp.symbols('m g km kf')  # Position, velocity, control
phi, theta, psi = sp.symbols('phi theta psi')
L = sp.symbols('L')
p, q, r = symbols('p q r')
F1, F2, F3, F4 = symbols('F1 F2 F3 F4')
F = Matrix([F1, F2, F3, F4])
I1, I2, I3 = symbols('I1 I2 I3')
I = Matrix([[I1, 0, 0], [0, I2, 0], [0, 0, I3]])
I_inv = I.inv()
px, py, pz, vx, vy, vz, ax, ay, az = symbols('px py pz vx vy vz ax ay az')
u1, u2, u3, u4 = symbols('u1 u2 u3 u4')
u = Matrix([u1, u2, u3, u4])
F1 = u1**2 *kf
F2 = u2**2 *kf
F3 = u3**2 *kf
F4 = u4**2 *kf
M1 = u1**2 *km
M2 = u2**2 *km
M3 = u3**2 *km
M4 = u4**2 *km

# the rotation matrix from the quadrotor's body frame to the world frame
R = sp.Matrix([[cos(psi)*cos(theta) - sin(phi)*sin(psi)*sin(theta), -cos(phi)*sin(psi), cos(psi)*sin(theta) + cos(theta)*sin(phi)*sin(psi)],
                 [cos(theta)*sin(psi) + cos(psi)*sin(phi)*sin(theta), cos(phi)*cos(psi), sin(psi)*sin(theta) - cos(psi)*cos(theta)*sin(phi)],
                 [-cos(phi)*sin(theta), sin(phi), cos(phi)*cos(theta)]])

# positional acceleration equation
acc_matrix = (1/m) * ((Matrix([0, 0, -m*g]) + R@Matrix([0, 0, F1 + F2 + F3 + F4])))
# angular acceleration equation
ang_matrix = I_inv @ (Matrix([[L*(F2-F4)], [L*(F3-F1)], [M1 - M2 + M3 - M4]]) - (Matrix([[0, p, q], [-p , 0, r], [-q, -r, 0]])@(I@Matrix([p, q, r]))))


In [15]:

print(f"acceleration matrix with shape {acc_matrix.shape}")
pprint(acc_matrix)

acceleration matrix with shape (3, 1)
⎡                                       ⎛     2        2        2        2⎞ ⎤
⎢(sin(φ)⋅sin(ψ)⋅cos(θ) + sin(θ)⋅cos(ψ))⋅⎝kf⋅u₁  + kf⋅u₂  + kf⋅u₃  + kf⋅u₄ ⎠ ⎥
⎢────────────────────────────────────────────────────────────────────────── ⎥
⎢                                    m                                      ⎥
⎢                                                                           ⎥
⎢                                        ⎛     2        2        2        2⎞⎥
⎢(-sin(φ)⋅cos(ψ)⋅cos(θ) + sin(ψ)⋅sin(θ))⋅⎝kf⋅u₁  + kf⋅u₂  + kf⋅u₃  + kf⋅u₄ ⎠⎥
⎢───────────────────────────────────────────────────────────────────────────⎥
⎢                                     m                                     ⎥
⎢                                                                           ⎥
⎢                ⎛     2        2        2        2⎞                        ⎥
⎢         -g⋅m + ⎝kf⋅u₁  + kf⋅u₂  + kf⋅u₃  + kf⋅u₄ ⎠⋅cos(φ)⋅cos(θ)          ⎥
⎢         ────────────────

In [16]:
print(f"angular acceleration matrix with shape {acc_matrix.shape}")
pprint(ang_matrix)

angular acceleration matrix with shape (3, 1)
⎡                           ⎛     2        2⎞       ⎤
⎢      -I₂⋅p⋅q - I₃⋅q⋅r + L⋅⎝kf⋅u₂  - kf⋅u₄ ⎠       ⎥
⎢      ──────────────────────────────────────       ⎥
⎢                        I₁                         ⎥
⎢                                                   ⎥
⎢           2       2     ⎛       2        2⎞       ⎥
⎢       I₁⋅p  - I₃⋅r  + L⋅⎝- kf⋅u₁  + kf⋅u₃ ⎠       ⎥
⎢       ─────────────────────────────────────       ⎥
⎢                         I₂                        ⎥
⎢                                                   ⎥
⎢                       2        2        2        2⎥
⎢I₁⋅p⋅q + I₂⋅q⋅r + km⋅u₁  - km⋅u₂  + km⋅u₃  - km⋅u₄ ⎥
⎢───────────────────────────────────────────────────⎥
⎣                         I₃                        ⎦


In [17]:
# This is the Jacobian of the acceleration matrix w.r.t. input u
C = acc_matrix.jacobian(u)
pprint(C)

⎡2⋅kf⋅u₁⋅(sin(φ)⋅sin(ψ)⋅cos(θ) + sin(θ)⋅cos(ψ))   2⋅kf⋅u₂⋅(sin(φ)⋅sin(ψ)⋅cos(θ
⎢──────────────────────────────────────────────   ────────────────────────────
⎢                      m                                                m     
⎢                                                                             
⎢2⋅kf⋅u₁⋅(-sin(φ)⋅cos(ψ)⋅cos(θ) + sin(ψ)⋅sin(θ))  2⋅kf⋅u₂⋅(-sin(φ)⋅cos(ψ)⋅cos(
⎢───────────────────────────────────────────────  ────────────────────────────
⎢                       m                                                m    
⎢                                                                             
⎢             2⋅kf⋅u₁⋅cos(φ)⋅cos(θ)                            2⋅kf⋅u₂⋅cos(φ)⋅
⎢             ─────────────────────                            ───────────────
⎣                       m                                                m    

) + sin(θ)⋅cos(ψ))   2⋅kf⋅u₃⋅(sin(φ)⋅sin(ψ)⋅cos(θ) + sin(θ)⋅cos(ψ))   2⋅kf⋅u₄⋅
──────────────────   ──────────────────────────────

In [18]:
# This is the Jacobian of the angular acceleration matrix w.r.t. the state angular velocities
D = ang_matrix.jacobian(Matrix([p, q, r]))
pprint(D)

⎡-I₂⋅q   -I₂⋅p - I₃⋅r   -I₃⋅q  ⎤
⎢──────  ────────────   ────── ⎥
⎢  I₁         I₁          I₁   ⎥
⎢                              ⎥
⎢2⋅I₁⋅p                -2⋅I₃⋅r ⎥
⎢──────       0        ────────⎥
⎢  I₂                     I₂   ⎥
⎢                              ⎥
⎢ I₁⋅q   I₁⋅p + I₂⋅r     I₂⋅q  ⎥
⎢ ────   ───────────     ────  ⎥
⎣  I₃         I₃          I₃   ⎦


In [19]:
# This is the Jacobian of the angular acceleration matrix w.r.t. input u
E = ang_matrix.jacobian(u)
pprint(E)

⎡             2⋅L⋅kf⋅u₂             -2⋅L⋅kf⋅u₄ ⎤
⎢     0       ─────────      0      ───────────⎥
⎢                 I₁                     I₁    ⎥
⎢                                              ⎥
⎢-2⋅L⋅kf⋅u₁              2⋅L⋅kf⋅u₃             ⎥
⎢───────────      0      ─────────       0     ⎥
⎢     I₂                     I₂                ⎥
⎢                                              ⎥
⎢  2⋅km⋅u₁    -2⋅km⋅u₂    2⋅km⋅u₃    -2⋅km⋅u₄  ⎥
⎢  ───────    ─────────   ───────    ───────── ⎥
⎣     I₃          I₃         I₃          I₃    ⎦


In [20]:
# Manually fill in our linearized A matrix
final_a_matrix = Matrix([
                [0,0,0,1,0,0,0,0,0,0,0,0],
                [0,0,0,0,1,0,0,0,0,0,0,0],
                [0,0,0,0,0,1,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,1,0,0],
                [0,0,0,0,0,0,0,0,0,1,0,0],
                [0,0,0,0,0,0,0,0,0,0,0,1],
                [0,0,0,0,0,0,0,0,0, *D.row(0)],
                [0,0,0,0,0,0,0,0,0, *D.row(1)],
                [0,0,0,0,0,0,0,0,0, *D.row(2)]])

# Manually fill in our linearized B matrix
final_b_matrix = Matrix([
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    C,
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    E
])

In [21]:
print(f"final_a_matrix (shape:({final_a_matrix.shape}))")
pprint(final_a_matrix)

final_a_matrix (shape:((12, 12)))
⎡0  0  0  1  0  0  0  0  0    0          0           0    ⎤
⎢                                                         ⎥
⎢0  0  0  0  1  0  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  1  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    1          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    1          0           0    ⎥
⎢                                                         ⎥
⎢0  0 

In [22]:
print(f"final_b_matrix (shape:({final_b_matrix.shape}))")
pprint(final_b_matrix)


final_b_matrix (shape:((12, 4)))
⎡                       0                                                0    
⎢                                                                             
⎢                       0                                                0    
⎢                                                                             
⎢                       0                                                0    
⎢                                                                             
⎢2⋅kf⋅u₁⋅(sin(φ)⋅sin(ψ)⋅cos(θ) + sin(θ)⋅cos(ψ))   2⋅kf⋅u₂⋅(sin(φ)⋅sin(ψ)⋅cos(θ
⎢──────────────────────────────────────────────   ────────────────────────────
⎢                      m                                                m     
⎢                                                                             
⎢2⋅kf⋅u₁⋅(-sin(φ)⋅cos(ψ)⋅cos(θ) + sin(ψ)⋅sin(θ))  2⋅kf⋅u₂⋅(-sin(φ)⋅cos(ψ)⋅cos(
⎢───────────────────────────────────────────────  ────────────────────────────
⎢                  

In [23]:
# Here we substitute in our symbolic values with our actual model parameters
values: Dict[symbols, float] = {
    L: 0.0397,
    I1: 2.3951e-5,
    I2: 2.3951e-5,
    I3: 3.2347e-5,
    px: 10, py: 10, pz: 10,
    vx: 0, vy: 0, vz: 0,
    ax: 0, ay: 0, az: 0,
    # x: 0, v: 0, u: 0,
    p: 0., q: 0., r: 0.,
    phi: 0.0, theta: 0.0, psi: 0.0,
    g: -9.81,
    km: 7.94e-12,
    kf: 3.16e-10,
    u1: 25.0, u2: 26.0, u3: 27.0, u4: 28.0,
    # u1: 0, u2: 0, u3: 0, u4: 0,
    F1: u1**2 *kf,
    F2: u2**2 *kf,
    F3: u3**2 *kf,
    F4: u4**2 *kf,
    m: 0.027
}

# Substitute values into the matrix
final_a_matrix_numeric = final_a_matrix.subs(values)
final_b_matrix_numeric = final_b_matrix.subs(values)


In [24]:
print(f"final_a_matrix with substituted values: ")
pprint(final_a_matrix_numeric)

final_a_matrix with substituted values: 
⎡0  0  0  1  0  0  0  0  0  0  0  0⎤
⎢                                  ⎥
⎢0  0  0  0  1  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  1  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  1  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  1  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  1⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎣0  0  0  0  0  0  0  0  0  0  0  0⎦


In [25]:
print(f"final_b_matrix with substituted values: ")
pprint(final_b_matrix_numeric)

final_b_matrix with substituted values: 
⎡         0                     0                    0                    0   
⎢                                                                             
⎢         0                     0                    0                    0   
⎢                                                                             
⎢         0                     0                    0                    0   
⎢                                                                             
⎢         0                     0                    0                    0   
⎢                                                                             
⎢         0                     0                    0                    0   
⎢                                                                             
⎢5.85185185185185e-7   6.08592592592593e-7        6.32e-7        6.55407407407
⎢                                                                             
⎢         0

In [26]:
# Convert to a NumPy array
final_a_matrix_numpy = np.array(final_a_matrix_numeric.evalf(), dtype=np.float32)
final_b_matrix_numpy = np.array(final_b_matrix_numeric.evalf(), dtype=np.float32)

In [27]:
print(f"final_a_matrix (shape : {final_a_matrix_numeric.shape})")
pprint(final_a_matrix_numpy)


final_a_matrix (shape : (12, 12))
 [[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [28]:
print(f"final_b_matrix (shape : {final_b_matrix_numeric.shape})")
pprint(final_b_matrix_numpy)

final_b_matrix (shape : (12, 4))
 [[ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 5.8518521e-07  6.0859259e-07  6.3200002e-07  6.5540740e-07]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  2.7236876e-05  0.0000000e+00 -2.9332019e-05]
  [-2.6189304e-05  0.0000000e+00  2.8284447e-05  0.0000000e+00]
 [ 1.2273163e-05 -1.2764090e-05  1.3255016e-05 -1.3745943e-05]]


In [29]:
def get_controllability_matrix(a: ndarray, b: ndarray) -> ndarray:
    """
    For an LTI system, return its controllability matrix.
    :param a: Linear state equations w.r.t. current state.
    :param b: Linear state equations w.r.t. input.
    :return:
    """
    n = a.shape[1]  # state dimension
    m = b.shape[1]  # control dimension

    # Use Cayley-Hamilton theorem to create a (n, nxm) controllability matrix whose rank tells us
    # how many states in the system are controllable.
    ctrl_matrix = b
    for i in range(1, n):
        ctrl_matrix = np.hstack((ctrl_matrix, np.linalg.matrix_power(a, i)@b))
    assert ctrl_matrix.shape == (n, n*m), f"Controllability matrix does not have the proper shape of ({(n, n*m)})"
    return ctrl_matrix

def controllability_rank(c: ndarray) -> int:
    """
    Gets the rank of the controllability matrix which tells us how many states in the system are controllable.
    :param c:   Controllability matrix
    :return:    Rank
    """
    rank_c = np.linalg.matrix_rank(c)
    return rank_c

def get_controllable_subsystem(a: ndarray, b: ndarray, c: ndarray, rank_c: int) -> Tuple[ndarray, ndarray, ndarray]:
    """
    Performs Kalman decomposition to get the controllable subsystem as well as the linear transformation matrix T which puts the original system into this modal form.
    :param a:       Linear state dynamics w.r.t. current state.
    :param b:       Linear state dynamics w.r.t. input.
    :param c:       Controllability matrix.
    :param rank_c:  Rank of the controllability matrix.
    :return:
    """

    # Use QR decomposition to find an orthonormal basis for the controllable subspace
    Q, _ = np.linalg.qr(c)  # Q has orthonormal columns spanning the controllable subspace
    T = Q  # Transformation matrix

    # Transform A and B
    A_transformed = np.linalg.inv(T) @ a @ T
    B_transformed = np.linalg.inv(T) @ b
    print(f"New state system: ")
    print(f"Transformed A: \n{A_transformed}")
    print(f"Transformed B: \n{B_transformed}")
    # Identify which states are controllable
    controllable_states = np.where(np.sum(np.abs(T), axis=1) > 1e-6)[0]
    uncontrollable_states = np.setdiff1d(np.arange(A.shape[0]), controllable_states)
    print(f"Controllable states: {controllable_states}")
    print(f"Uncontrollable states: {uncontrollable_states}")

    # Extract controllable part
    A_c = A_transformed[:rank_c, :rank_c]
    B_c = B_transformed[:rank_c, :]

    return A_c, B_c, T

In [30]:
A_prev = final_a_matrix_numpy
# Adjust this value if needed (try 1e-5, 1e-7, etc.)
A = A_prev
B = final_b_matrix_numpy
Q = np.eye(12)
R = np.eye(4)
print("Condition number of A:", np.linalg.cond(A))
print("Condition number of B:", np.linalg.cond(B))
ctrl_matrix = get_controllability_matrix(A, B)
ctrl_rank = controllability_rank(ctrl_matrix)
can_ctrl = ctrl_rank >= A.shape[1]
print(f"Is this linear system controllable? {can_ctrl}")
print(f"Controllability matrix has a rank of {ctrl_rank} for a system with {A.shape[1]} state dimensions.")

# get the controllable subsystem:
A_c, B_c, T = get_controllable_subsystem(A, B, ctrl_matrix, ctrl_rank)

Condition number of A: inf
Condition number of B: 32.38606
Is this linear system controllable? False
Controllability matrix has a rank of 7 for a system with 12 state dimensions.
New state system: 
Transformed A: 
[[-4.84479657e-27  3.15290447e-25  1.21967103e-25 -1.70483452e-24
   1.28045601e-34 -1.23474749e-32  1.15309554e-24 -3.16041951e-18
  -2.32163658e-16  1.19328427e-16  1.06766683e-16  2.37335508e-16]
 [-1.05609278e-26  1.98199947e-25  5.92827984e-26 -3.03192526e-26
   3.59614315e-35 -1.62693576e-32 -6.98324601e-23  1.91397682e-16
  -6.47747469e-17  3.30566056e-17  3.20829942e-17  6.52952354e-17]
 [ 2.54332044e-18  9.67415851e-18  9.97101956e-18 -3.60192754e-17
   1.86432467e-33 -7.99175410e-33  1.11788098e-23 -3.06390247e-17
  -1.65076253e-16  8.48814827e-17  7.55759125e-17  1.68889807e-16]
 [ 2.16151145e-17  2.53147013e-17  6.07056139e-17  1.06745908e-15
  -5.31510762e-32 -3.59914153e-31 -1.34446930e-21  3.68493849e-15
  -2.77562829e-15  1.42204815e-15  1.32087250e-15  2.8195

In [31]:
print(f"Transformation matrix T (shape ({T.shape})): \n{T}")

Transformation matrix T (shape ((12, 12))): 
[[ 0.0000000e+00  8.3266727e-17  2.7755576e-16  4.4157958e-15
  -7.1200274e-18  4.8284740e-19  4.6854649e-09 -1.2841982e-02
   8.1752398e-05  5.4597268e-03  9.1081786e-01 -4.1257256e-01]
 [-0.0000000e+00  0.0000000e+00 -6.0715322e-17 -3.6394499e-15
   3.8332981e-18 -2.2132712e-18  5.1554527e-10 -1.4130130e-03
  -1.3817938e-02 -9.1351455e-01  1.7227109e-01  3.6826748e-01]
 [-0.0000000e+00  0.0000000e+00  0.0000000e+00 -1.0616508e-15
  -4.7625959e-02  3.1512436e-02  9.9836802e-01  3.6425985e-07
  -6.2536781e-16  2.2204460e-16  2.7755576e-16  8.0491169e-16]
 [-0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
   3.4739551e-19 -3.4307591e-17 -1.3002971e-09  3.5638688e-03
  -6.2984759e-01  3.2371667e-01  2.8979713e-01  6.4382023e-01]
 [-0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
   0.0000000e+00  6.6570013e-17  3.6482251e-07 -9.9991018e-01
  -2.2264186e-03  2.3745913e-03 -1.0908308e-02  7.0730089e-03]
 [-2.0228742e-02 -2.

In [32]:
print(f"A_c (shape ({A_c.shape})): \n{A_c}")
print(f"B_c (shape ({B_c.shape})): \n{B_c}")

A_c (shape ((7, 7))): 
[[-4.8447966e-27  3.1529045e-25  1.2196710e-25 -1.7048345e-24
   1.2804560e-34 -1.2347475e-32  1.1530955e-24]
 [-1.0560928e-26  1.9819995e-25  5.9282798e-26 -3.0319253e-26
   3.5961431e-35 -1.6269358e-32 -6.9832460e-23]
 [ 2.5433204e-18  9.6741585e-18  9.9710196e-18 -3.6019275e-17
   1.8643247e-33 -7.9917541e-33  1.1178810e-23]
 [ 2.1615114e-17  2.5314701e-17  6.0705614e-17  1.0674591e-15
  -5.3151076e-32 -3.5991415e-31 -1.3444693e-21]
 [ 4.2474192e-01 -3.5226393e-01  8.3396912e-01  3.0409657e-09
  -4.9937406e-18  7.5469999e-18 -2.9786281e-17]
 [-5.8842903e-11 -1.3021154e+00 -5.5000633e-01  3.1516843e-02
   1.6895019e-18 -3.9306502e-18  5.3303380e-17]
 [-3.7737058e-10  3.7527141e-09  3.2499319e-09 -1.0004975e+00
   5.0915313e-17 -3.3225109e-17 -8.8651575e-16]]
B_c (shape ((7, 4))): 
[[-2.8928403e-05  5.4029806e-06  1.9969963e-05  5.8185947e-06]
 [ 4.3473447e-13 -2.9596411e-05  9.3491362e-06  2.2114065e-05]
 [ 5.2435980e-13 -6.0052435e-13 -2.2133661e-05  2.2803863

At this point, we have the equations that we need in order to produce a linear state-feedback controller for our controllable subsystem in the form:
$$
u = -Kx
$$
Finding this controller involves solving the LQR (Linear Quadratic Regular) control optimal problem:
$$
\min_u J(x, u) := \min_u \int_{t=0}^{\infty} x^\top Qx + u^\top R u \; dt
$$
where $Q \in \mathbb{R}^{n\times n}$ and $R \in \mathbb{R}^{m\times m}$. We also have that $Q$ and $R$ are both symmetric. $Q$ is positive semi-definite and $R$ is positive definite.

Luckily, this optimization problem has a closed-form solution given by the algebraic Riccati equation, and scipy has a method that can solve for this solution.

In [33]:
Q_c = np.eye(ctrl_rank)
R_c = np.eye(B_c.shape[1])
S = scipy.linalg.solve_continuous_are(A_c, B_c, Q_c, R_c)
K = -np.linalg.solve(R_c, B_c.T @ S)
print(f"final_a_matrix (shape : {final_a_matrix_numeric.shape})\n{final_a_matrix_numeric}")
print(f"K (shape : {K.shape})\n{K}")

final_a_matrix (shape : (12, 12))
Matrix([[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
K (shape : (4, 7))
[[ 6.92300445e+01 -3.42159043e+01  1.43855894e+02  6.14518791e+02
   4.95368506e-01 -7.28620656e-03 -4.64972525e-01]
 [-4.79361520e+01  2.38033210e+02 -1.03701876e+01  6.43909046e+02
  -4.89993135e-01 -7.06243464e-01 -5.11005769e-01]
 [ 7.33336540e+01 -3.72150432e+01  1.55695726e+02  6.61164507e+02
   5.34988972e-01 -7.86619883e-03 -5.02078112e-01]
 [-4.55962709e+01 -1.11849314e+02 -1.52744635e+02  6.14462697e+02
  -4.77811229e-01  7.07887847e-01 -5.20183838e-01]]


In [34]:
from control import ctrb; 
# C = ctrb(A,B); 
# Q, _ = np.linalg.qr(C); 
# T = np.hstack((Q[:,:7], scipy.linalg.null_space(C.T)))
# print(T)
A = np.eye(12)
C = ctrb(A, B); 
rank = np.linalg.matrix_rank(C)
print(rank)

C = ctrb(A, B)  # 12 × 48

# QR decomposition
Q, R = np.linalg.qr(C)
Q_c = Q[:, :7]  # 12 × 7, controllable subspace

# Print Q_c to see significant entries
print(Q_c)

4
[[ 0.00000000e+00  8.32667268e-17  2.77555756e-16  4.41579597e-15
  -1.00000000e+00  3.10838398e-18  1.84634873e-17]
 [-0.00000000e+00  0.00000000e+00 -6.07153217e-17 -3.63944985e-15
  -7.69783542e-18  1.00000000e+00 -7.36222816e-17]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00 -1.06165077e-15
   7.37257477e-18  0.00000000e+00  1.00000000e+00]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.78141682e-18 -1.38043730e-16  3.14182829e-17]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  2.77555756e-17  5.55111512e-17]
 [-2.02287410e-02 -2.42559161e-02 -5.70506170e-02 -9.97871573e-01
  -4.43048376e-15 -3.64632587e-15 -1.07000194e-15]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e

# Next Steps

1) We want to be able to concretely say which states are controllable. It does not matter that 7 states are controllable we cannot control for example, the altitude of the quadrotor. We need to answer this question.
2) $K$ is our linear state-feedback controller. Our controller can be written as $u = -Kx$. In other words, based on our current state $x$, $u$ will give us the control input we should use to move towards equilibrium. **However**, we only have $K$ for the 7 state sub-system. Can we write what our control would look like for our original 12 state-system? Is it necessary to rewrite K? 
3) Once we know which states are controllable, and we are satisfied with our controller, we should create some plot or visualization to quickly confirm that this controller can stabilize our system for points *near* the equilibrium point. 